# Part 2 实操：TinyCNN 训练 EuroSAT RGB

这个 notebook 使用本地数据：

```text
data/eurosat/EuroSAT_RGB/
```

定位说明：这是一个 **TinyCNN baseline on EuroSAT RGB subset**，用于教学演示从 `Conv2d` 到遥感场景分类的完整流程。默认使用 3000 张均衡子集、训练 30 轮；不同电脑速度不同，可以在训练配置单元里修改 `MAX_SAMPLES`、`EPOCHS` 和 `BATCH_SIZE`。

流程：

1. 检查数据集
2. 可视化样例图
3. 定义 Dataset
4. 定义 TinyCNN
5. 分层划分训练/验证集
6. 训练模型
7. 保存模型和训练日志
8. 绘制训练曲线和混淆矩阵

## 0. 如果还没安装依赖

在 notebook 里可以取消下面代码的注释运行一次。  
如果你已经装好 PyTorch，就跳过。

In [ ]:
# !pip install torch pillow matplotlib

## 1. 导入包和基础配置

In [ ]:
import csv
import json
import random
from pathlib import Path

from PIL import Image

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, random_split


def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

## 2. 数据路径和类别

In [ ]:
def find_repo_root(start=None):
    """从当前目录向上找包含 data/eurosat/EuroSAT_RGB 的仓库根目录。"""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]

    # 如果你从 notebook 所在目录启动，Path.cwd() 可能是 notebooks/algorithms。
    # 这里逐级向上找，直到找到本地数据目录。
    for candidate in candidates:
        data_root = candidate / "data" / "eurosat" / "EuroSAT_RGB"
        if data_root.exists():
            return candidate

    raise FileNotFoundError(
        "没有找到 data/eurosat/EuroSAT_RGB。\n"
        f"当前工作目录是: {Path.cwd()}\n"
        "请确认数据位于 F:/遥感转码基础知识/data/eurosat/EuroSAT_RGB，"
        "或者先在仓库根目录启动 Jupyter。"
    )


REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "data" / "eurosat" / "EuroSAT_RGB"
OUT_DIR = REPO_ROOT / "outputs" / "part2_cnn_notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "AnnualCrop",
    "Forest",
    "HerbaceousVegetation",
    "Highway",
    "Industrial",
    "Pasture",
    "PermanentCrop",
    "Residential",
    "River",
    "SeaLake",
]

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("data root:", DATA_ROOT)
print("exists:", DATA_ROOT.exists())
print("output dir:", OUT_DIR)

## 3. 检查每个类别的图片数量

In [ ]:
class_counts = {}
missing_or_empty = []

for class_name in CLASS_NAMES:
    class_dir = DATA_ROOT / class_name
    files = sorted(class_dir.glob("*.jpg"))
    class_counts[class_name] = len(files)
    print(f"{class_name:22s} {len(files):5d}  {class_dir}")
    if len(files) == 0:
        missing_or_empty.append(class_dir)

print("\ntotal images:", sum(class_counts.values()))

if missing_or_empty:
    raise FileNotFoundError(
        "这些类别目录没有找到 jpg 图片：\n"
        + "\n".join(str(p) for p in missing_or_empty)
    )

## 4. 看每个类别的一张样例图

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for ax, class_name in zip(axes.ravel(), CLASS_NAMES):
    files = sorted((DATA_ROOT / class_name).glob("*.jpg"))
    if not files:
        raise FileNotFoundError(f"{DATA_ROOT / class_name} 下面没有 .jpg 图片")

    image_path = files[0]
    image = Image.open(image_path).convert("RGB")
    ax.imshow(image)
    ax.set_title(class_name, fontsize=9)
    ax.axis("off")

fig.tight_layout()
sample_grid_path = OUT_DIR / "eurosat_samples.png"
fig.savefig(sample_grid_path, dpi=160)
plt.show()

print("saved:", sample_grid_path)

## 5. PIL 图像转 Tensor

这里不用 `torchvision`，避免环境里没有 torchvision 时跑不起来。  
EuroSAT RGB 图像是 `64x64x3`，转换后是 PyTorch 需要的 `[C, H, W]`。

In [ ]:
EUROSAT_MEAN = torch.tensor([0.3444, 0.3809, 0.4082]).view(3, 1, 1)
EUROSAT_STD = torch.tensor([0.2037, 0.1366, 0.1148]).view(3, 1, 1)


def pil_to_tensor(image):
    # bytearray creates a writable copy, avoiding torch.frombuffer non-writable warnings.
    raw = torch.tensor(bytearray(image.tobytes()), dtype=torch.uint8)
    tensor = raw.view(image.size[1], image.size[0], 3).permute(2, 0, 1).float()
    return tensor / 255.0


test_image_path = sorted((DATA_ROOT / CLASS_NAMES[0]).glob("*.jpg"))[0]
test_image = Image.open(test_image_path).convert("RGB")
test_tensor = pil_to_tensor(test_image)
print("PIL size:", test_image.size)
print("tensor shape:", test_tensor.shape)
print("min/max:", test_tensor.min().item(), test_tensor.max().item())

## 6. 定义 EuroSAT Dataset

In [ ]:
class EuroSATRGBDataset(Dataset):
    def __init__(self, root, train=True, image_size=64):
        self.root = Path(root)
        self.train = train
        self.image_size = image_size
        self.class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
        self.samples = []

        for class_name in CLASS_NAMES:
            class_dir = self.root / class_name
            if not class_dir.exists():
                raise FileNotFoundError(f"Missing class directory: {class_dir}")
            for path in sorted(class_dir.glob("*.jpg")):
                self.samples.append((path, self.class_to_idx[class_name]))

        if not self.samples:
            raise RuntimeError(f"No jpg images found under {self.root}")

    def __len__(self):
        return len(self.samples)

    def _augment(self, image):
        if self.train:
            if random.random() < 0.5:
                image = image.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
            if random.random() < 0.5:
                image = image.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

        if image.size != (self.image_size, self.image_size):
            image = image.resize((self.image_size, self.image_size), Image.Resampling.BILINEAR)
        return image

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        image = self._augment(image)
        tensor = pil_to_tensor(image)
        tensor = (tensor - EUROSAT_MEAN) / EUROSAT_STD
        return tensor, torch.tensor(label, dtype=torch.long)


train_base = EuroSATRGBDataset(DATA_ROOT, train=True)
val_base = EuroSATRGBDataset(DATA_ROOT, train=False)

image, label = train_base[0]
print("dataset size:", len(train_base))
print("one image:", image.shape)
print("one label:", label.item(), CLASS_NAMES[label.item()])
print("class_to_idx:", train_base.class_to_idx)

## 7. 反归一化并可视化一个 batch

In [ ]:
def denormalize(tensor):
    return (tensor * EUROSAT_STD + EUROSAT_MEAN).clamp(0, 1)


preview_loader = DataLoader(Subset(train_base, list(range(20))), batch_size=10, shuffle=False)
images, labels = next(iter(preview_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, image, label in zip(axes.ravel(), images, labels):
    image = denormalize(image).permute(1, 2, 0).numpy()
    ax.imshow(image)
    ax.set_title(CLASS_NAMES[label.item()], fontsize=9)
    ax.axis("off")

fig.tight_layout()
plt.show()

## 8. 定义 TinyCNN

结构：

```text
Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d
Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d
Conv2d -> BatchNorm2d -> ReLU
AdaptiveAvgPool2d
Linear
```

In [ ]:
class TinyCNNClassifier(nn.Module):
    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = TinyCNNClassifier(num_classes=len(CLASS_NAMES)).to(device)
dummy = torch.randn(4, 3, 64, 64).to(device)
logits = model(dummy)
print("logits:", logits.shape)
print("num params:", sum(p.numel() for p in model.parameters()))

## 9. 打印每一层 shape

In [ ]:
shape_model = TinyCNNClassifier()
x = torch.randn(4, 3, 64, 64)
print("input", x.shape)

with torch.no_grad():
    y = x
    for layer in shape_model.features:
        y = layer(y)
        print(f"{layer.__class__.__name__:18s}", y.shape)
    y = shape_model.pool(y)
    print(f"{'AdaptiveAvgPool2d':18s}", y.shape)
    y = torch.flatten(y, 1)
    print(f"{'flatten':18s}", y.shape)
    y = shape_model.classifier(y)
    print(f"{'Linear':18s}", y.shape)

## 10. 分层划分训练集和验证集

第一次运行建议 `MAX_SAMPLES = 3000`。注意：EuroSAT 文件是按类别排序的，不能直接取前 3000 张，否则全是 `AnnualCrop`。

下面的代码会先按类别均衡抽样，再在每个类别内部切分 train/val。

可调参数：

- `MAX_SAMPLES = 3000`：默认每类约 300 张，适合快速演示。
- `MAX_SAMPLES = 0`：使用完整 27000 张，训练更慢但更正式。
- `BATCH_SIZE = 64`：显存不够可改成 32 或 16。
- `NUM_WORKERS = 0`：Windows/Jupyter 下最稳；脚本运行时可尝试调大。

In [ ]:
MAX_SAMPLES = 3000  # 0 表示使用全部数据
VAL_RATIO = 0.2
BATCH_SIZE = 64
NUM_WORKERS = 0


def make_balanced_stratified_split(dataset, max_samples=0, val_ratio=0.2, seed=42):
    rng = random.Random(seed)
    indices_by_class = {class_idx: [] for class_idx in range(len(CLASS_NAMES))}

    for idx, (_, label) in enumerate(dataset.samples):
        indices_by_class[label].append(idx)

    selected_indices = []
    if max_samples and max_samples > 0:
        per_class = max_samples // len(CLASS_NAMES)
        remainder = max_samples % len(CLASS_NAMES)
        for class_idx in range(len(CLASS_NAMES)):
            class_indices = indices_by_class[class_idx][:]
            rng.shuffle(class_indices)
            take = per_class + (1 if class_idx < remainder else 0)
            selected_indices.extend(class_indices[: min(take, len(class_indices))])
    else:
        for class_indices in indices_by_class.values():
            selected_indices.extend(class_indices)

    selected_by_class = {class_idx: [] for class_idx in range(len(CLASS_NAMES))}
    for idx in selected_indices:
        label = dataset.samples[idx][1]
        selected_by_class[label].append(idx)

    train_indices = []
    val_indices = []
    for class_idx, class_indices in selected_by_class.items():
        rng.shuffle(class_indices)
        val_count = max(1, int(round(len(class_indices) * val_ratio))) if class_indices else 0
        val_indices.extend(class_indices[:val_count])
        train_indices.extend(class_indices[val_count:])

    rng.shuffle(train_indices)
    rng.shuffle(val_indices)
    return train_indices, val_indices


train_indices, val_indices = make_balanced_stratified_split(
    train_base,
    max_samples=MAX_SAMPLES,
    val_ratio=VAL_RATIO,
    seed=42,
)

train_dataset = Subset(train_base, train_indices)
val_dataset = Subset(val_base, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

print("train images:", len(train_dataset))
print("val images:", len(val_dataset))

train_label_counts = torch.bincount(
    torch.tensor([train_base.samples[i][1] for i in train_indices]),
    minlength=len(CLASS_NAMES),
)
val_label_counts = torch.bincount(
    torch.tensor([val_base.samples[i][1] for i in val_indices]),
    minlength=len(CLASS_NAMES),
)

print("\ntrain class counts:")
for name, count in zip(CLASS_NAMES, train_label_counts.tolist()):
    print(f"{name:22s} {count:5d}")

print("\nval class counts:")
for name, count in zip(CLASS_NAMES, val_label_counts.tolist()):
    print(f"{name:22s} {count:5d}")

images, labels = next(iter(train_loader))
print("\nbatch images:", images.shape)
print("batch labels:", labels.shape)

## 11. 训练和验证函数

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return {
        "loss": total_loss / total_count,
        "acc": total_correct / total_count,
    }


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return {
        "loss": total_loss / total_count,
        "acc": total_correct / total_count,
    }

## 12. 先 overfit 32 张图

In [ ]:
# 每类取 4 张，共 40 张，用来测试模型能不能在很小的数据上记住样本。
small_indices = []
for class_idx in range(len(CLASS_NAMES)):
    class_indices = [idx for idx, (_, label) in enumerate(train_base.samples) if label == class_idx]
    small_indices.extend(class_indices[:4])

small_dataset = Subset(train_base, small_indices)
small_loader = DataLoader(small_dataset, batch_size=20, shuffle=True, num_workers=0)

set_seed(123)
small_model = TinyCNNClassifier(num_classes=len(CLASS_NAMES)).to(device)
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=1e-3)

small_history = []
for epoch in range(1, 4):
    train_metrics = train_one_epoch(small_model, small_loader, small_optimizer, device)
    small_history.append({"epoch": epoch, **train_metrics})
    print(f"epoch {epoch:02d} | loss {train_metrics['loss']:.4f} acc {train_metrics['acc']:.3f}")

## 13. 正式训练 TinyCNN

默认训练 `EPOCHS = 30`，这是为了让曲线更完整，适合展示仓库实操效果。

不同电脑训练速度不同：

- 想快速试跑：把 `EPOCHS` 改成 `3` 或 `5`。
- 想得到更稳定曲线：保持 `30`。
- 想跑完整数据：前面把 `MAX_SAMPLES` 改成 `0`，这里可以先用 `EPOCHS = 10` 起步。

请注意：这张曲线应表述为 **TinyCNN baseline on EuroSAT RGB subset**，不是完整 EuroSAT SOTA 精度。

In [ ]:
EPOCHS = 30
LR = 1e-3
EXPERIMENT_NOTE = "TinyCNN baseline on EuroSAT RGB subset"

set_seed(42)
model = TinyCNNClassifier(num_classes=len(CLASS_NAMES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history = []
best_val_acc = -1.0
OUT_DIR.mkdir(parents=True, exist_ok=True)
best_path = OUT_DIR / "best_tiny_cnn.pt"

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
    }
    history.append(row)

    if row["val_acc"] > best_val_acc:
        best_val_acc = row["val_acc"]
        best_path.parent.mkdir(parents=True, exist_ok=True)
        checkpoint = {
            "model_state_dict": model.state_dict(),
            "class_names": CLASS_NAMES,
            "best_val_acc": best_val_acc,
            "config": {
                "max_samples": MAX_SAMPLES,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "lr": LR,
                "note": EXPERIMENT_NOTE,
            },
        }
        # PyTorch 2.0 on Windows may mis-handle non-ASCII Path strings.
        # Saving through an opened file object works reliably.
        with best_path.open("wb") as f:
            torch.save(checkpoint, f)

    print(
        f"epoch {epoch:02d} | "
        f"train loss {row['train_loss']:.4f} acc {row['train_acc']:.3f} | "
        f"val loss {row['val_loss']:.4f} acc {row['val_acc']:.3f}"
    )

print("experiment:", EXPERIMENT_NOTE)
print("best val acc:", best_val_acc)
print("saved:", best_path)

## 14. 保存训练日志

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
history_path = OUT_DIR / "history.csv"

with history_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)

class_to_idx_path = OUT_DIR / "class_to_idx.json"
class_to_idx_path.write_text(
    json.dumps({name: idx for idx, name in enumerate(CLASS_NAMES)}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("history:", history_path)
print("class_to_idx:", class_to_idx_path)

## 15. 绘制训练曲线

In [ ]:
epochs = [row["epoch"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
axes[0].plot(epochs, [row["val_loss"] for row in history], label="val")
axes[0].set_title("loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, [row["train_acc"] for row in history], label="train")
axes[1].plot(epochs, [row["val_acc"] for row in history], label="val")
axes[1].set_title("accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

fig.suptitle("TinyCNN baseline on EuroSAT RGB subset", fontsize=12)
fig.tight_layout()
OUT_DIR.mkdir(parents=True, exist_ok=True)
curve_path = OUT_DIR / "training_curve.png"
fig.savefig(curve_path, dpi=160)
plt.show()

print("saved:", curve_path)

## 16. 混淆矩阵

In [ ]:
@torch.no_grad()
def compute_confusion_matrix(model, dataloader, device, num_classes):
    matrix = torch.zeros(num_classes, num_classes, dtype=torch.long)
    model.eval()

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)
        preds = model(images).argmax(dim=1)

        for target, pred in zip(labels.cpu(), preds.cpu()):
            matrix[target, pred] += 1

    return matrix


cm = compute_confusion_matrix(model, val_loader, device, len(CLASS_NAMES))
cm

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(cm.numpy(), cmap="Blues")

ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8)
ax.set_yticklabels(CLASS_NAMES, fontsize=8)
ax.set_xlabel("predicted")
ax.set_ylabel("target")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()

OUT_DIR.mkdir(parents=True, exist_ok=True)
cm_path = OUT_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=160)
plt.show()

print("saved:", cm_path)

## 17. 看每类验证准确率

In [ ]:
per_class_total = cm.sum(dim=1)
per_class_correct = cm.diag()
per_class_acc = per_class_correct.float() / per_class_total.clamp(min=1).float()

for class_name, acc, correct, total in zip(CLASS_NAMES, per_class_acc, per_class_correct, per_class_total):
    print(f"{class_name:22s} acc={acc.item():.3f} ({correct.item()}/{total.item()})")

## 18. 下一步可以怎么改

这个 notebook 默认是 **TinyCNN baseline on EuroSAT RGB subset**。为了兼顾不同用户电脑性能，核心参数都可以直接改：

- 快速试跑：`MAX_SAMPLES = 1000`，`EPOCHS = 3`，`BATCH_SIZE = 32`
- 默认展示：`MAX_SAMPLES = 3000`，`EPOCHS = 30`，`BATCH_SIZE = 64`
- 更正式实验：`MAX_SAMPLES = 0`，`EPOCHS = 10-30`

还可以继续尝试：

- 把通道数 `16, 32, 64` 改成 `32, 64, 128`。
- 尝试去掉 `BatchNorm2d`，观察训练是否变慢。
- 对比 `Adam` 和 `SGD(momentum=0.9)`。
- 后续再加入 ResNet 迁移学习，与 TinyCNN baseline 对比。

## 19. 生成仓库宣传图

训练或数据检查完成后，可以运行下面的脚本生成适合 README、朋友圈、公众号或课程展示的图片。

输出目录：

```text
outputs/part2_cnn_promo/
```

In [ ]:
# 在 notebook 里运行这一格即可重新生成宣传图
!python ../../scripts/part2_cnn/make_promo_figures.py